# RDDT ATTR - Specialty Tier agents

- Step 1: session & table QC
- Step 2: wide net (Tier 1 + Tier 2) → `ATTR_WIDE_NET_CANDIDATES`
- Step 3: evidence temps `ATTR_EVID_*` for candidates only
- Step 4 (previous): Ortho / Cardio / Neuro Tier 1–2 + combo shortlist (≥2 specialties), hardcoded rules, run in Snowflake SQL
- Step 4 v2 (new, config-driven): same result, but the Tier 1/2 clinical rules live in editable JSON files instead of Python - see `specialty_configs/v2/atoms`, `specialty_configs/v2/buckets`, `specialty_configs/v2/features`
- **Session TEMPORARY tables only** - no permanent tables, no stage saves

**Table and column names are configured in cell 1.2 of this notebook.** Nothing else needs editing when the client/data engineer renames a table or column.

**Upload the entire `specialty_configs/` folder into Snowflake notebook Files** (keep `v1/` and `v2/` intact). It contains everything Step 3 and both Step 4 engines need:
- `rddt_attr_sql.py` - Steps 1–3 (shared by both engines)
- `v1/` - hardcoded Step 4 engine: `rddt_specialty_config.py` + `rddt_specialty_sql.py` (cells 4.0/4.1)
- `v2/` - config-driven Step 4 engine: `loader.py`, `sql_generator.py`, `atoms/`, `buckets/`, `features/` (cells 4.3/4.4)


### Step 1 - Session & table QC

### 1.1 Active session

In [ ]:
import importlib
import sys
from pathlib import Path

import pandas as pd
from snowflake.snowpark.context import get_active_session

# Shared Steps 1–3 live in specialty_configs/rddt_attr_sql.py
HERE = Path.cwd()
CANDIDATE_DIRS = [
    HERE / "specialty_configs",
    HERE,
    Path("/tmp/specialty_configs"),
    Path("/tmp"),
]
_mod_dir = next((p for p in CANDIDATE_DIRS if (p / "rddt_attr_sql.py").exists()), None)
if _mod_dir is None:
    raise FileNotFoundError(
        "rddt_attr_sql.py not found. Searched:\n  " + "\n  ".join(str(p) for p in CANDIDATE_DIRS)
    )
if str(_mod_dir) not in sys.path:
    sys.path.insert(0, str(_mod_dir))
print("rddt_attr_sql.py loaded from:", _mod_dir)

import rddt_attr_sql as rsql
importlib.reload(rsql)

session = get_active_session()
session


In [ ]:
# Change to match our environment
DATABASE = "RDDT"
SCHEMA = "PUBLIC"

session.sql(f"USE DATABASE {DATABASE}").collect()
session.sql(f"USE SCHEMA {SCHEMA}").collect()
print("Current database:", session.get_current_database())
print("Current schema:", session.get_current_schema())

### 1.2 TABLE CONFIGURATION - edit here only

Every table and column name used anywhere downstream comes from this dict.

- `name` - physical table name (add `DB.SCHEMA.` prefix, or set `namespace`, if they live elsewhere)
- `columns` - logical name → physical column name; identifiers are quoted, so spaces and `/` are fine
- `enabled` - set `False` for a table we are not using (Excel marked **no need**)
- `required` - this table is **needed for our analysis**
- `required_columns` - fields we marked **need** in the Excel dictionary. Those columns must exist in the delivery.

In [ ]:
SOURCE_CONFIG = {
    # Set to "MY_DB.MY_SCHEMA" to qualify every table, or None to use the current schema.
    # required / required_columns = Excel analysis "need" in the data dictionary
    "namespace": None,
    "tables": {
        #------ Census  (need: Member/PatientId, BirthDate, Gender, City, State, FamilyId)
        "census": {
            "name": "CENSUS",
            "enabled": True,
            "required": True,
            "columns": {
                "patient_id": "Member/PatientId",
                "birth_date": "BirthDate",
                "gender": "Gender",
                "city": "City",
                "state": "State",
                "family_id": "FamilyId",
            },
            "required_columns": (
                "patient_id", "birth_date", "gender", "city", "state", "family_id",
            ),
        },
        # ------ Encounter/Visit
        "encounter": {
            "name": "ENCOUNTER_VISIT",
            "enabled": True,
            "required": True,
            "columns": {
                "encounter_id": "EncounterId/VisitId",
                "patient_id": "Member/PatientId",
                "encounter_date": "Encounter/Visit Date",
            },
            "required_columns": ("encounter_id", "patient_id", "encounter_date"),
        },
        # ----- Claim
        "claim": {
            "name": "CLAIM",
            "enabled": True,
            "required": True,
            "columns": {
                "patient_id": "Member/PatientId",
                "encounter_id": "EncounterId/VisitId",
                "diagnosis_code": "DiagnosisCode",
                "other_diagnosis_9": "OtherDiagnosisCodes9",
                "other_diagnosis_10": "OtherDiagnosisCodes10",
                "procedure_code": "ProcedureCode",
                "procedure_modifier_1": "ProcedureModifier1",
                "procedure_modifier_2": "ProcedureModifier2",
                "procedure_modifier_3": "ProcedureModifier3",
                "diagnosis_type": "DiagnosisType",
                "provider_type": "ProviderType",
                "specialty_code": "SpecialtyCode",
                "specialty_name": "SpecialtyName",
                "drg_code": "DRGCode",
                "clinical_notes": "ClinicalNotes",
                "from_date": "FromDate",
                "to_date": "ToDate",
            },
            "diagnosis_columns": ("diagnosis_code", "other_diagnosis_9", "other_diagnosis_10"),
            "procedure_columns": ("procedure_code",),
            "required_columns": (
                "patient_id", "encounter_id",
                "diagnosis_code", "other_diagnosis_9", "other_diagnosis_10",
                "procedure_code", "procedure_modifier_1", "procedure_modifier_2", "procedure_modifier_3",
                "diagnosis_type", "provider_type", "specialty_code", "specialty_name",
                "drg_code", "clinical_notes", "from_date", "to_date",
            ),
        },
        # ----- Lab
        "lab": {
            "name": "LAB",
            "enabled": True,
            "required": True,
            "columns": {
                "patient_id": "Member/PatientId",
                "encounter_id": "EncounterId/VisitId",
                "lab_id": "LabId",
                "lab_request_id": "LabRequestId",
                "lab_result_id": "LabResultId",
                "observation_identifier": "ObservationIdentifier",
                "observation_value": "ObservationValue",
                "result_status": "ObservationResultStatus",
                "observation_datetime": "ObservationDateTime",
                "lab_result_note": "LabResultNote",
            },
            "required_columns": (
                "lab_id", "encounter_id", "patient_id",
                "lab_request_id", "lab_result_id",
                "observation_identifier", "observation_value", "result_status",
                "observation_datetime", "lab_result_note",
            ),
        },
        # ----- Medical History
        "medical_history": {
            "name": "MEDICAL_HISTORY",
            "enabled": True,
            "required": True,
            "columns": {
                "patient_id": "Member/PatientId",
                "encounter_id": "EncounterId/VisitId",
                "record_id": "MedicalHistoryId",
                "source_category": "Source/Category",
                "value": "Value",
                "snomed": "SNOMED",
                "secondary_snomed": "Secondary SNOMED",
                "event_date": "Date",
            },
            "required_columns": (
                "record_id", "encounter_id", "patient_id",
                "source_category", "value", "snomed", "secondary_snomed", "event_date",
            ),
        },
        # ----- Surgical History
        "surgical_history": {
            "name": "SURGICAL_HISTORY",
            "enabled": True,
            "required": True,
            "columns": {
                "patient_id": "Member/PatientId",
                "encounter_id": "EncounterId/VisitId",
                "record_id": "SurgicalHistoryId",
                "source_category": "Source/Category",
                "value": "Value",
                "snomed": "SNOMED",
                "secondary_snomed": "Secondary SNOMED",
                "event_date": "Date",
            },
            "required_columns": (
                "record_id", "encounter_id", "patient_id",
                "source_category", "value", "snomed", "secondary_snomed", "event_date",
            ),
        },
        # ----- Family History
        "family_history": {
            "name": "FAMILY_HISTORY",
            "enabled": True,
            "required": True,
            "columns": {
                "patient_id": "Member/PatientId",
                "encounter_id": "EncounterId/VisitId",
                "record_id": "FamilyHistoryId",
                "condition": "Condition",
                "status": "Status",
                "family_member": "FamilyMember",
                "snomed": "SNOMED",
                "event_date": "Date",
            },
            "required_columns": (
                "record_id", "encounter_id", "patient_id",
                "snomed", "condition", "status", "family_member", "event_date",
            ),
        },
        # ----- Clinical Note
        "clinical_note": {
            "name": "CLINICAL_NOTE",
            "enabled": True,
            "required": True,
            "columns": {
                "patient_id": "Member/PatientId",
                "encounter_id": "EncounterId/VisitId",
                "note_id": "NoteId",
                "note_type": "NoteType",
                "note_text": "Clinical Note Text",
                "event_date": "Date",
            },
            "required_columns": (
                "note_id", "encounter_id", "patient_id",
                "note_type", "note_text", "event_date",
            ),
        },
        # ----- Social History  (no need)
        "social_history": {
            "name": "SOCIAL_HISTORY",
            "enabled": False,
            "required": False,
            "columns": {
                "patient_id": "Member/PatientId",
                "encounter_id": "EncounterId/VisitId",
                "value": "Value",
                "event_date": "Date",
            },
            "required_columns": (),
        },
        # ----- Medication  (no need)
        "medication": {
            "name": "MEDICATION",
            "enabled": False,
            "required": False,
            "columns": {
                "patient_id": "Member/PatientId",
                "encounter_id": "EncounterId/VisitId",
                "medication_name": "Medication Name",
                "status": "Medication Status",
                "event_date": "Date",
            },
            "required_columns": (),
        },
    },
}

# Keyword ILIKE nets only. SNOMED stays in SOURCE_CONFIG for a later code filter,
# not for English phrase search (codes look like 35488005, not "carpal tunnel").
TEXT_COLUMNS = {
    "lab": ("observation_identifier", "observation_value", "lab_result_note"),
    "medical_history": ("value", "source_category"),
    "surgical_history": ("value", "source_category"),
    "family_history": ("condition",),
    "clinical_note": ("note_text",),
    "social_history": ("value",),
}

for logical, source in SOURCE_CONFIG["tables"].items():
    flag = "on " if source.get("enabled", True) else "off"
    print(f"[{flag}] {logical:18s} -> {source['name']}")


### 1.3 Validate the configuration against the real tables

Fails immediately if an **analysis-need** table or `need` column is missing, so a rename shows up here instead of as a wrong patient count later.

In [ ]:
validation = rsql.validate_sources(session, SOURCE_CONFIG, raise_on_error=True)
validation

### 1.4 Total patients

In [ ]:
print(f"Total Patient Count: {rsql.patient_count(session, SOURCE_CONFIG):,}")

### 1.5 QC for each configured table - null patient id + total rows

In [ ]:
qc = rsql.table_qc(session, SOURCE_CONFIG)
qc

### Step 2 - Wide net (Tier 1 + Tier 2)

### Why specialty buckets?
- Easy to add a new cardiology code without touching neuro lists
- Easy to see coverage gaps per organ system
- Matches our ATTR framework agent sheets

### Why the table/column map?
- Shows exactly which columns we search today
- Column names come from cell 1.2, so a rename never touches the clinical lists below

### 2.1 Code dictionary by specialty (with explanations)

Each entry: `code` → meaning. Prefix `LIKE` patterns listed separately per specialty.

Claim data often stores ICD codes without dots, so matching tries both `G56.03` and `G5603` automatically.

In [ ]:
# =============================================================================
# ATTR TIER 1 + TIER 2 CODE BOOK (by specialty)
# Add new codes inside the right specialty block only.
# =============================================================================

ATTR_CODES = {
    # -------------------------------------------------------------------------
    "orthopedic_msk": {
        "tier": "1-2",
        "exact_icd": {
            # Carpal tunnel (Tier 1 - bilateral G56.03 BEST)
            "G56.00": "Carpal tunnel syndrome, unspecified upper limb",
            "G56.01": "Carpal tunnel syndrome, right upper limb",
            "G56.02": "Carpal tunnel syndrome, left upper limb",
            "G56.03": "Carpal tunnel syndrome, bilateral upper limbs (BEST T1)",
            # Trigger finger (Tier 2)
            "M65.30": "Trigger finger, unspecified finger",
            "M65.311": "Trigger finger, right index",
            "M65.312": "Trigger finger, left index",
            "M65.319": "Trigger finger, unspecified index",
            "M65.321": "Trigger finger, right middle",
            "M65.322": "Trigger finger, left middle",
            "M65.329": "Trigger finger, unspecified middle",
            "M65.331": "Trigger finger, right ring",
            "M65.332": "Trigger finger, left ring",
            "M65.339": "Trigger finger, unspecified ring",
            "M65.341": "Trigger finger, right little",
            "M65.342": "Trigger finger, left little",
            "M65.349": "Trigger finger, unspecified little",
            "M65.351": "Trigger finger, right thumb",
            "M65.352": "Trigger finger, left thumb",
            "M65.359": "Trigger finger, unspecified thumb",
            # Lumbar stenosis (Tier 2)
            "M48.06": "Spinal stenosis, lumbar region",
            "M48.061": "Spinal stenosis, lumbar, without neurogenic claudication",
            "M48.062": "Spinal stenosis, lumbar, with neurogenic claudication (BEST)",
            # Spontaneous tendon rupture - biceps proxy (Tier 1)
            "M66.821": "Spontaneous rupture of other tendons, right upper arm",
            "M66.822": "Spontaneous rupture of other tendons, left upper arm",
            "M66.829": "Spontaneous rupture of other tendons, unspecified upper arm",
            # Biceps injury proxies (verify locally - coding inconsistent)
            "S46.111A": "Strain of muscle/tendon long head biceps, right, initial",
            "S46.112A": "Strain of muscle/tendon long head biceps, left, initial",
            "S46.119A": "Strain of muscle/tendon long head biceps, unspecified, initial",
        },
        "exact_cpt": {
            "64721": "Neuroplasty median nerve at carpal tunnel (CTS release) - T1",
        },
        "like_prefixes": [
            "G56.0%",   # all CTS
            "M65.3%",   # all trigger finger
            "M48.06%",  # lumbar stenosis family
            "M66.82%",  # spontaneous tendon rupture upper arm
        ],
        "notes": "Ortho clustering (CTS+trigger+stenosis) is Step 4 composite, not required here.",
    },

    # -------------------------------------------------------------------------
    "cardiology": {
        "tier": "1-2",
        "exact_icd": {
            # HFpEF / diastolic HF (Tier 2)
            "I50.30": "Unspecified diastolic (congestive) heart failure",
            "I50.31": "Acute diastolic (congestive) heart failure",
            "I50.32": "Chronic diastolic (congestive) heart failure (common HFpEF code)",
            "I50.33": "Acute on chronic diastolic (congestive) heart failure",
            "I50.9": "Heart failure, unspecified (broader net - noisier)",
            # Cardiomyopathy / thick-wall proxies (Tier 1-2)
            "I42.0": "Dilated cardiomyopathy",
            "I42.1": "Obstructive hypertrophic cardiomyopathy (elderly HCM = ATTR phenocopy risk)",
            "I42.2": "Other hypertrophic cardiomyopathy",
            "I42.5": "Other restrictive cardiomyopathy (Tier 2 restrictive pattern)",
            "I42.8": "Other cardiomyopathies",
            "I42.9": "Cardiomyopathy, unspecified",
            "I51.7": "Cardiomegaly (weak LVH/thick-wall PROXY - Tier 1 support only)",
            # Aortic stenosis (Tier 2)
            "I35.0": "Nonrheumatic aortic (valve) stenosis",
            "I35.2": "Nonrheumatic aortic stenosis with insufficiency",
            # AF / flutter (Tier 2)
            "I48.0": "Paroxysmal atrial fibrillation",
            "I48.1": "Persistent atrial fibrillation",
            "I48.2": "Chronic atrial fibrillation",
            "I48.3": "Typical atrial flutter",
            "I48.4": "Atypical atrial flutter",
            "I48.91": "Unspecified atrial fibrillation",
            "I48.92": "Unspecified atrial flutter",
            # Conduction (Tier 2)
            "I44.0": "Atrioventricular block, first degree",
            "I44.1": "Atrioventricular block, second degree",
            "I44.2": "Atrioventricular block, complete",
            "I44.30": "Unspecified atrioventricular block",
            "I44.39": "Other atrioventricular block",
            "I45.10": "Unspecified right bundle-branch block",
            "I45.19": "Other right bundle-branch block",
            # Device (Tier 2)
            "Z95.0": "Presence of cardiac pacemaker",
            # Weak ECG proxy
            "R94.31": "Abnormal electrocardiogram (weak - low-voltage needs NLP)",
        },
        "exact_cpt": {
            "33361": "TAVR/TAVI - replacement aortic valve",
            "33362": "TAVR/TAVI related",
            "33363": "TAVR/TAVI related",
            "33364": "TAVR/TAVI related",
            "33365": "TAVR/TAVI related",
            "33366": "TAVR/TAVI related",
            "33367": "TAVR/TAVI related",
            "33368": "TAVR/TAVI related",
            "33369": "TAVR/TAVI related",
        },
        "like_prefixes": [
            "I50.3%",  # diastolic / HFpEF family
            "I42.%",   # cardiomyopathy family
            "I48.%",   # AF/flutter family
            "I44.%",   # AV block family
            "I35.%",   # aortic valve disease family
            "Z95.%",   # cardiac devices
        ],
        "notes": "Apical sparing, ECV, T1, LGE, PYP grade, voltage-mass are mostly NOT structured ICD - now mainly Clinical Note NLP.",
    },

    # -------------------------------------------------------------------------
    "neurology_autonomic": {
        "tier": "1-2",
        "exact_icd": {
            "G62.9": "Polyneuropathy, unspecified (Tier 2)",
            "G62.89": "Other specified polyneuropathies",
            "G60.8": "Other hereditary and idiopathic neuropathies (SFN bridge)",
            "G60.9": "Hereditary and idiopathic neuropathy, unspecified",
            "G61.81": "CIDP (Tier 1 mislabel pathway)",
            "I95.1": "Orthostatic hypotension (Tier 2 autonomic)",
            "N52.9": "Male erectile dysfunction, unspecified (Tier 2 autonomic support)",
            "R63.4": "Abnormal weight loss (pathway support with neuropathy)",
        },
        "exact_cpt": {},
        "like_prefixes": [
            "G62.%",
            "G60.%",
        ],
        "notes": "Refractory-to-IVIG and axonal EMG pattern need notes/meds - Clinical Note now",
    },

    # -------------------------------------------------------------------------
    "gastroenterology": {
        "tier": "1-2",
        "exact_icd": {
            "K31.84": "Gastroparesis (Tier 2)",
            "R68.81": "Early satiety (Tier 2)",
            "R19.7": "Diarrhea, unspecified (Tier 2)",
            "K59.00": "Constipation, unspecified",
            "K59.01": "Slow transit constipation",
            "K59.09": "Other constipation",
        },
        "exact_cpt": {},
        "like_prefixes": ["K59.0%"],
        "notes": "Alternating diarrhea+constipation has no single ICD - NLP phrases feed Step 4. GI+PN composite = Step 4.",
    },

    # -------------------------------------------------------------------------
    "urology": {
        "tier": "2",
        "exact_icd": {
            "R33.9": "Retention of urine, unspecified (Tier 2)",
        },
        "exact_cpt": {},
        "like_prefixes": [],
        "notes": "Elevated PVR is NLP/numeric - often not coded.",
    },

    # -------------------------------------------------------------------------
    "ophthalmology": {
        "tier": "1-2",
        "exact_icd": {
            "H43.391": "Other vitreous opacities, right eye (Tier 1 ATTRv-leaning)",
            "H43.392": "Other vitreous opacities, left eye",
            "H43.393": "Other vitreous opacities, bilateral",
            "H43.399": "Other vitreous opacities, unspecified eye",
            "H40.89": "Other specified glaucoma (Tier 2 secondary glaucoma bridge)",
            "H57.00": "Unspecified anomaly of pupillary function (Tier 2 weak)",
        },
        "exact_cpt": {},
        "like_prefixes": ["H43.39%"],
        "notes": "Vitreous amyloid wording is NLP-preferred when present.",
    },

    # -------------------------------------------------------------------------
    "family_history_heme_gate": {
        "tier": "1-2",
        "exact_icd": {
            "Z82.41": "Family history of sudden cardiac death (Tier 1/2 FHx cardiac red flag)",
            "D47.2": "Monoclonal gammopathy (MGUS) - AL SAFETY GATE flag, NOT ATTR proof",
        },
        "exact_cpt": {},
        "like_prefixes": [],
        "notes": "D47.2 keeps patient in net for later ATTR-vs-AL routing; do not score as ATTR+.",
    },
}


def flatten_exact_codes(code_book):
    """Merge all specialty exact ICD+CPT into one set for SQL IN lists."""
    icd, cpt = set(), set()
    for block in code_book.values():
        icd.update(block.get("exact_icd", {}).keys())
        cpt.update(block.get("exact_cpt", {}).keys())
    return sorted(icd), sorted(cpt)


def flatten_like_prefixes(code_book):
    prefixes = []
    for block in code_book.values():
        prefixes.extend(block.get("like_prefixes", []))
    seen, out = set(), []
    for p in prefixes:
        if p not in seen:
            seen.add(p)
            out.append(p)
    return out


for specialty, block in ATTR_CODES.items():
    print("=" * 70)
    print(f"SPECIALTY: {specialty}  (tier {block.get('tier')})")
    print("- ICD -")
    for code, meaning in block.get("exact_icd", {}).items():
        print(f"  {code:12s}  {meaning}")
    if block.get("exact_cpt"):
        print("- CPT -")
        for code, meaning in block["exact_cpt"].items():
            print(f"  {code:12s}  {meaning}")
    if block.get("like_prefixes"):
        print("- LIKE prefixes -")
        for p in block["like_prefixes"]:
            print(f"  {p}")
    if block.get("notes"):
        print(f"NOTE: {block['notes']}")
    print()

target_icd_exact, target_cpt_exact = flatten_exact_codes(ATTR_CODES)
like_prefixes = flatten_like_prefixes(ATTR_CODES)
print(
    f"TOTAL exact ICD: {len(target_icd_exact)} | "
    f"exact CPT: {len(target_cpt_exact)} | LIKE prefixes: {len(like_prefixes)}"
)

### 2.2 Table - column NLP / text map (what we search)

| v1.1 Table | Columns used for ATTR text / values | Role in Step 2 |
|------------|-------------------------------------|----------------|
| **Claim** | `DiagnosisCode`, `OtherDiagnosisCodes9/10`, `ProcedureCode`, `ClinicalNotes` | Structured ICD/CPT (primary) + claim note text |
| **Encounter_Visit** | `EncounterId/VisitId`, `Encounter/Visit Date` | Visit dates for the code timeline |
| **Census** | `Member/PatientId`, `Gender`, `BirthDate`, `FamilyId` | Demographics (not a Step-2 net by itself) |
| **Medical History** | `Value`, `Source/Category` | Keyword net (`SNOMED` kept for a later code filter only) |
| **Surgical History** | `Value`, `Source/Category` | CTS release, stenosis surgery, TAVR, etc. |
| **Family History** | `Condition`, `FamilyMember` | Sudden death / CM / neuropathy / amyloid FHx |
| **Lab** | `ObservationIdentifier`, `ObservationValue`, `LabResultNote` | Congo red / amyloid / TTR / FLC / NT-proBNP / PYP text |
| **Clinical Note** | `Clinical Note Text`, `NoteType` | echo/CMR/EMG narrative, "suspect amyloid" |
| **Social History** | `Value` | Low ATTR yield; disabled by default (not used) |
| **Medication** | `Medication Name`, `Medication Status` | Not Used |

### 2.3 Data gaps - need to look and see existing data after data set available

| Needed clinical artifact | Why (Tier 1/2) | current status |
|--------------------------|----------------|-------------|
| Echo report / GLS / wall thickness mm | Apical sparing, reduced GLS | **now possible** via `Clinical Note` (imaging reports) |
| Cardiac MRI reports (ECV, T1, LGE) | Tier 1 | **now possible** via `Clinical Note` |
| ECG over-read text (low voltage, voltage-mass) | Tier 1 | **Possible** via `Clinical Note`; `R94.31` weak |
| PYP/DPD report with Perugini grade | Tier 1 | **Possible** - note/lab text, no structured grade |
| Medication / Rx history (IVIG, HF drug stops) | Tier 1-2 | **Now delivered** (`Medication`) - not used (but applicable) |
| Clinical notes / consults | Tier 1-2 | `Clinical Note` |
| Autonomic lab / tilt / QSART | Confirms neurogenic OH | **No** unless in `Lab` |
| Genetics report module | Pathogenic TTR variants | NLP in labs/notes |
| Vitals longitudinal (BP fall) | BP paradox | **No** vitals table |


In [ ]:
# Machine-readable gap list (for docs / future mapping)
# after data availble we need to wire this
FUTURE_DATA_GAPS = [
    {"artifact": "Echo GLS / speckle-tracking / wall thickness mm", "tier": "1-2",
     "in_v11_schema": "partial", "workaround_today": "Clinical Note imaging reports + ICD proxies I42.*, I51.7"},
    {"artifact": "Cardiac MRI (ECV, T1, LGE)", "tier": "1",
     "in_v11_schema": "partial", "workaround_today": "Clinical Note imaging reports"},
    {"artifact": "ECG narrative (low voltage / voltage-mass)", "tier": "1",
     "in_v11_schema": "partial", "workaround_today": "Clinical Note; R94.31 weak"},
    {"artifact": "PYP/DPD report with Perugini grade", "tier": "1",
     "in_v11_schema": "partial", "workaround_today": "LabResultNote / Clinical Note keywords"},
    {"artifact": "Medications / infusion (IVIG, HF drug stops)", "tier": "1-2",
     "in_v11_schema": True, "workaround_today": "Medication table delivered - not wired into the net yet"},
    {"artifact": "Full clinical notes", "tier": "1-2",
     "in_v11_schema": True, "workaround_today": "Clinical Note Text (new in v1.1)"},
    {"artifact": "Genetics structured module", "tier": "1",
     "in_v11_schema": "partial", "workaround_today": "NLP in labs/notes + E85 later"},
    {"artifact": "Vitals time series", "tier": "2-3",
     "in_v11_schema": False, "workaround_today": None},
     ]

for g in FUTURE_DATA_GAPS:
    print(f"- {g['artifact']} | in_v11={g['in_v11_schema']} | workaround={g['workaround_today']}")

### 2.4 Claim - patients with specialty ICD/CPT targets

Scans every configured diagnosis column for ICD (exact + `LIKE`) and every procedure column for CPT, in both dotted and undotted form.

In [ ]:
claim_ids = rsql.code_net_patient_ids(
    session,
    SOURCE_CONFIG,
    exact_icd=target_icd_exact,
    exact_cpt=target_cpt_exact,
    like_prefixes=like_prefixes,
)
print(f"CLAIM unique patients: {len(claim_ids):,}")

In [ ]:
# Per-ICD unique patients across all diagnosis columns.
# Same patient can appear under many codes
icd_counts_df = rsql.per_code_counts(
    session, SOURCE_CONFIG, target_icd_exact, code_kind="icd"
)
icd_counts_df

In [ ]:
# Per-CPT unique patients across all procedure columns.
cpt_counts_df = rsql.per_code_counts(
    session, SOURCE_CONFIG, target_cpt_exact, code_kind="cpt"
)
cpt_counts_df

### 2.5 Lab - ObservationIdentifier / ObservationValue / LabResultNote

In [ ]:
LAB_TERMS = [

    # ================================================================
    # GENERAL / CROSS-SPECIALTY - Tier 1 (amyloid biopsy confirmation)
    # Used by: Ortho sig 3, GI sig 6, Nephro sig 3, Agent5 sig 5/6
    # ================================================================

    # Tier 1 - Congo red stain / apple-green birefringence (the definitive
    # amyloid stain, any specialty's biopsy)
    ("[PATH T1] congo red", "%congo red%"),
    ("[PATH T1] apple-green", "%apple-green%"),
    ("[PATH T1] apple green", "%apple green%"),
    ("[PATH T1] birefringence", "%birefringence%"),

    # Tier 1 - amyloid / transthyretin / ATTR, general mentions
    ("[PATH T1] amyloid", "%amyloid%"),
    ("[PATH T1] transthyretin", "%transthyretin%"),
    ("[PATH T1] attr (spaced)", "% attr %"),
    ("[PATH T1] attr-", "%attr-%"),

    # Tier 1 - vitreous amyloid (Ophthalmology sig 1)
    ("[EYE T1] vitreous amyloid", "%vitreous amyloid%"),
    ("[EYE T1] amyloid vitreous", "%amyloid vitreous%"),
    ("[EYE T1] vitreous amyloidosis", "%vitreous amyloidosis%"),

    # Tier 1 - mass spectrometry TTR typing (Agent5 sig 6, definitive typing)
    ("[PATH T1] mass spectrometry", "%mass spectrometry%"),
    ("[PATH T1] LC-MS/MS", "%LC-MS/MS%"),


    # ================================================================
    # CARDIOLOGY
    # ================================================================

    # Tier 1 - bone scintigraphy / Perugini grade (Agent2 sig 8 - the single
    # most consequential cardiac imaging finding)
    ("[CARDIO T1] pyrophosphate", "%pyrophosphate%"),
    ("[CARDIO T1] perugini", "%perugini%"),
    ("[CARDIO T1] PYP", "%PYP%"),
    ("[CARDIO T1] PYP scan", "%PYP scan%"),

    # Tier 1 - apical sparing / "cherry on top" strain pattern (sig 1)
    ("[CARDIO T1] apical sparing", "%apical sparing%"),
    ("[CARDIO T1] cherry on top", "%cherry on top%"),
    ("[CARDIO T1] cherry-on-top", "%cherry-on-top%"),

    # Tier 1 - expanded extracellular volume, ECV (sig 2)
    ("[CARDIO T1] extracellular volume", "%extracellular volume%"),

    # Tier 1 - elevated native T1 mapping (sig 3)
    ("[CARDIO T1] native T1 (bounded)", ["% native T1 %", "native T1%", "%native T1", "% native T1%", "%native T1 %"]),

    # Tier 1 - explicit cardiologist suspicion of infiltrative cardiomyopathy/ amyloidosis (sig 5)
    ("[CARDIO T1] infiltrative cardiomyopathy", "%infiltrative cardiomyopathy%"),
    ("[CARDIO T1] cardiac amyloid", "%cardiac amyloid%"),

    # Tier 1 - voltage-mass mismatch / low QRS voltage relative to wall thickness (sig 7)
    ("[CARDIO T1] low QRS voltage", "%low QRS voltage%"),
    ("[CARDIO T1] voltage-mass", "%voltage-mass%"),

    # Tier 2 - reduced global longitudinal strain, GLS (sig 11)
    ("[CARDIO T2] global longitudinal strain", "%global longitudinal strain%"),
    ("[CARDIO T2] GLS (bounded)", ["% GLS %", "GLS%", "%GLS", "% GLS%", "%GLS %"]),

    # Tier 3 - NT-proBNP / troponin (NOT Tier 1/2 in the source framework -
    # kept here as supporting lab evidence
    ("[CARDIO T3] nt-probnp", "%nt-probnp%"),
    ("[CARDIO T3] ntprobnp", "%ntprobnp%"),
    ("[CARDIO T3] troponin", "%troponin%"),


    # ================================================================
    # NEUROLOGY
    # ================================================================

    # Tier 2 - small fiber neuropathy (SFN), spelling variants (sig 2)
    ("[NEURO T2] small fiber neuropathy", "%small fiber neuropathy%"),
    ("[NEURO T2] small-fiber neuropathy", "%small-fiber neuropathy%"),
    ("[NEURO T2] small-fibre neuropathy", "%small-fibre neuropathy%"),

    # Tier 2 - reduced intraepidermal nerve fiber density, IENFD (sig 3)
    ("[NEURO T2] IENFD", "%IENFD%"),
    ("[NEURO T2] intraepidermal nerve", "%intraepidermal nerve%"),

    # Tier 2 - axonal pattern on EMG/NCS (sig 8)
    ("[NEURO T2] axonal neuropathy", "%axonal neuropathy%"),
    ("[NEURO T2] axonal polyneuropathy", "%axonal polyneuropathy%"),

    # Tier 1 - CIDP, especially refractory to IVIG/steroids (sig 9)
    ("[NEURO T1] CIDP", "%CIDP%"),

    # Tier 1 - neurologist explicitly suspects amyloid neuropathy / ATTR (sig 20)
    ("[NEURO T1] amyloid neuropathy", "%amyloid neuropathy%"),
    ("[NEURO T1] ATTR neuropathy", "%ATTR neuropathy%"),

    # Support (cross-signal) - EMG/NCS testing performed, used by several
    # neuro signals above rather than one specific tier on its own
    ("[NEURO support] nerve conduction", "%nerve conduction%"),
    ("[NEURO support] EMG (bounded)", ["% EMG %", "EMG%", "%EMG", "% EMG%", "%EMG %"]),

    # Tier 2 - orthostatic hypotension confirmed by autonomic testing (sig 12)
    ("[NEURO T2] orthostatic hypotension", "%orthostatic hypotension%"),

    # Tier 3 - abnormal tilt-table test (sig 13) / QSART & sudomotor testing
    # (sig 14). Kept in the net for completeness, but note these are Tier 3
    # in the source framework, not Tier 2 like orthostatic hypotension above.
    ("[NEURO T3] tilt table", "%tilt table%"),
    ("[NEURO T3] tilt-table", "%tilt-table%"),
    ("[NEURO T3] QSART", "%QSART%"),
    ("[NEURO T3] sudomotor", "%sudomotor%"),


    # ================================================================
    # HEMATOLOGY / AL SAFETY GATE (flag only - NOT ATTR-positive evidence)
    # ================================================================

    # Tier 1 (gate) - IFE / SPEP / UPEP / FLC triad (sig 1 gatekeeper,
    # sig 3 soft flag on abnormal FLC ratio alone)
    ("[HEME T1] free light chain", "%free light chain%"),
    ("[HEME T1] immunofixation", "%immunofixation%"),
    ("[HEME T1] SPEP", "%SPEP%"),
    ("[HEME T1] UPEP", "%UPEP%"),
    ("[HEME T1] kappa/lambda", "%kappa/lambda%"),
    ("[HEME T1] kappa lambda", "%kappa lambda%"),

    # Tier 1 (hard stop) - positive monoclonal protein (sig 2 - blocks the
    # non-invasive ATTR pathway outright, routes to biopsy instead)
    ("[HEME T1] monoclonal protein", "%monoclonal protein%"),
    ("[HEME T1] Bence Jones", "%Bence Jones%"),
    ("[HEME T1] M-spike", "%M-spike%"),
    ("[HEME T1] M spike", "%M spike%"),
    ("[HEME T1] paraprotein", "%paraprotein%"),
    ("[HEME T1] monoclonal band", "%monoclonal band%"),

    # Tier 1 (tie-breaker) - bone marrow biopsy showing clonal plasma cells (sig 4)
    ("[HEME T1] clonal plasma cells", "%clonal plasma cells%"),
    ("[HEME T1] plasma cell dyscrasia", "%plasma cell dyscrasia%"),


    # ================================================================
    # GENETICS
    # ================================================================

    # Tier 1 - pathogenic/likely-pathogenic TTR gene variant (sig 1)
    ("[GENE T1] TTR gene", "%TTR gene%"),
    ("[GENE T1] TTR mutation", "%TTR mutation%"),
    ("[GENE T1] transthyretin gene", "%transthyretin gene%"),

    # Tier 2 - specific founder mutations by ancestry (sig 2, 3, 4)
    ("[GENE T2] Val30Met", "%Val30Met%"),
    ("[GENE T2] Val50Met", "%Val50Met%"),
    ("[GENE T2] Val122Ile", "%Val122Ile%"),
    ("[GENE T2] V122I", "%V122I%"),
    ("[GENE T2] Thr60Ala", "%Thr60Ala%"),

    # Support (cross-signal) - bare "TTR" wide net
    ("[GENE support] TTR (bounded)", ["% TTR %", "TTR%", "% TTR%", "%TTR %"]),


    # ================================================================
    # GASTROENTEROLOGY - Tier 2
    # ================================================================

    # Tier 2 - gastroparesis / early satiety / delayed gastric emptying (sig 1)
    ("[GI T2] gastroparesis", "%gastroparesis%"),
    ("[GI T2] early satiety", "%early satiety%"),
    ("[GI T2] postprandial", "%postprandial%"),
    ("[GI T2] delayed gastric emptying", "%delayed gastric emptying%"),

    # Tier 2 - alternating constipation and diarrhea (sig 2)
    ("[GI T2] alternating constipation", "%alternating constipation%"),
    ("[GI T2] alternating diarrhea", "%alternating diarrhea%"),
    ("[GI T2] alternating bowel", "%alternating bowel%"),

    # Tier 2 - chronic/persistent diarrhea, later becoming continuous (sig 3)
    ("[GI T2] chronic diarrhea", "%chronic diarrhea%"),
    ("[GI T2] persistent diarrhea", "%persistent diarrhea%"),
    ("[GI T2] continuous diarrhea", "%continuous diarrhea%"),
]

# All patterns from the term list drive the wide net, so the net and the
# per-term QC counts can never drift apart.
LAB_PATTERNS = [p for _, pats in LAB_TERMS for p in (pats if isinstance(pats, list) else [pats])]

lab_ids = rsql.text_net_patient_ids(
    session, SOURCE_CONFIG, "lab", TEXT_COLUMNS["lab"], LAB_PATTERNS
)

In [ ]:
lab_term_counts_df = rsql.term_counts(
    session, SOURCE_CONFIG, "lab", TEXT_COLUMNS["lab"], LAB_TERMS
)
print(f"\nCombined unique (net cell): {len(lab_ids):,}")
lab_term_counts_df

### 2.6 Surgical History - Value / Source-Category

Organized by specialty → tier. Any hit makes the patient a candidate; Step 4 maps the text to `cts_*` features.

Note: `ILIKE '_'` is a single-char wildcard, so avoid bare `%joint_replacement%`.

In [ ]:
SURGICAL_TERMS = [

    # ================================================================
    # ORTHOPEDIC
    # ================================================================

    # Tier 1 - bilateral carpal tunnel syndrome / CTS release (sig 1)
    ("[ORTHO T1] bilateral carpal tunnel", "%bilateral carpal tunnel%"),
    ("[ORTHO T1] bilateral carpal_tunnel", "%bilateral carpal_tunnel%"),
    ("[ORTHO T1] bilateral carpal-tunnel", "%bilateral carpal-tunnel%"),
    ("[ORTHO T1] bilateral CTS", "%bilateral CTS%"),
    ("[ORTHO T1] carpal tunnel bilateral", "%carpal tunnel bilateral%"),
    ("[ORTHO T1] CTS bilateral", "%CTS bilateral%"),
    ("[ORTHO T1] carpal tunnel release", "%carpal tunnel release%"),
    ("[ORTHO T1] CTS release", "%CTS release%"),

    # Tier 1 - spontaneous distal biceps tendon rupture / "Popeye sign" (sig 2)
    ("[ORTHO T1] biceps", "%biceps%"),
    ("[ORTHO T1] popeye", "%popeye%"),

    # Tier 1/2 wide net - any carpal tunnel mention (unilateral falls back to
    # Tier 2 unless clustering - see ortho_05_clustering composite rule)
    ("[ORTHO T1-2] carpal tunnel (space)", "%carpal tunnel%"),
    ("[ORTHO T1-2] carpal_tunnel", "%carpal_tunnel%"),
    ("[ORTHO T1-2] carpal-tunnel", "%carpal-tunnel%"),
    ("[ORTHO T1-2] carpaltunnel", "%carpaltunnel%"),
    ("[ORTHO T1-2] CTS (bounded)", ["% CTS %", "CTS%", "%CTS", "% CTS%", "%CTS %"]),

    # Tier 2 - trigger finger / stenosing tenosynovitis (sig 6)
    ("[ORTHO T2] trigger finger", "%trigger finger%"),
    ("[ORTHO T2] trigger_finger", "%trigger_finger%"),
    ("[ORTHO T2] trigger-finger", "%trigger-finger%"),
    ("[ORTHO T2] trigger thumb", "%trigger thumb%"),
    ("[ORTHO T2] tenosynov", "%tenosynov%"),

    # Tier 2 - lumbar spinal stenosis (sig 7)
    ("[ORTHO T2] spinal stenosis", "%spinal stenosis%"),
    ("[ORTHO T2] lumbar stenosis", "%lumbar stenosis%"),
    ("[ORTHO T2] laminectomy", "%laminectomy%"),
    # ligamentum flavum specimen collected during laminectomy is the tissue actually checked for amyloid
    ("[ORTHO T2] ligamentum flavum", "%ligamentum flavum%"),

    # Support (cross-signal) - rotator cuff repair, one of the anatomic
    # buckets used by ortho_05_clustering
    ("[ORTHO support] rotator cuff", "%rotator cuff%"),

    # Tier 3 count-only - joint replacement / arthroplasty
    ("[ORTHO T3 count-only] joint replacement", "%joint replacement%"),
    ("[ORTHO T3 count-only] arthroplasty", "%arthroplasty%"),


    # ================================================================
    # CARDIOLOGY - Tier 2 (device / valve)
    # ================================================================

    # Tier 2 - permanent pacemaker OR ICD implanted before ATTR diagnosis (sig 16)
    ("[CARDIO T2] pacemaker", "%pacemaker%"),
    ("[CARDIO T2] defibrillator", "%defibrillator%"),
    ("[CARDIO T2] ICD (bounded)", ["% ICD %", "ICD%", "%ICD", "% ICD%", "%ICD %"]),
    ("[CARDIO T2] CRT-D", "%CRT-D%"),
    ("[CARDIO T2] CRT-P", "%CRT-P%"),

    # Tier 2 - low-flow, low-gradient aortic stenosis / TAVR workup (sig 12)
    ("[CARDIO T2] TAVR (bounded)", ["% TAVR %", "TAVR%", "%TAVR", "% TAVR%", "%TAVR %"]),
    ("[CARDIO T2] TAVI (bounded)", ["% TAVI %", "TAVI%", "%TAVI", "% TAVI%", "%TAVI %"]),
    ("[CARDIO T2] aortic valve", "%aortic valve%"),


    # ================================================================
    # NEUROLOGY
    # ================================================================

    # Support (cross-signal) - nerve biopsy, referenced by sig 20 (neurologist
    # explicitly suspects amyloid neuropathy / ATTR)
    ("[NEURO support] nerve biopsy", "%nerve biopsy%"),

    # Tier 2 - skin punch biopsy, the actual procedure used to collect tissue
    # for IENFD / reduced intraepidermal nerve fiber density (sig 3)
    ("[NEURO T2] skin biopsy", "%skin biopsy%"),


    # ================================================================
    # HEMATOLOGY - Tier 1 (tie-breaker)
    # ================================================================

    # bone marrow biopsy showing an isotype-matched clonal plasma
    # cell population (Agent4 sig 4)
    ("[HEME T1] bone marrow biopsy", "%bone marrow biopsy%"),
    ("[HEME T1] bone marrow aspiration", "%bone marrow aspiration%"),


    # ================================================================
    # NEPHROLOGY - Tier 1
    # ================================================================

    # renal biopsy showing amyloid deposits (sig 3) had zero
    # procedure-name coverage before. Fairly rare/specific procedure, so
    # low noise risk for a real gain.
    ("[NEPHRO T1] renal biopsy", "%renal biopsy%"),
    ("[NEPHRO T1] kidney biopsy", "%kidney biopsy%"),


    # ================================================================
    # OPHTHALMOLOGY
    # ================================================================

    # vitrectomy (sig 1, Tier 1) - one of the highest-value signals in
    # the whole framework ("close to a specific finding for hereditary ATTR")
    ("[EYE T1] vitrectomy", "%vitrectomy%"),

    # trabeculectomy, the procedure family behind secondary open-angle
    # glaucoma (sig 2, Tier 2).
    ("[EYE T2] trabeculectomy", "%trabeculectomy%"),
]

# Tier 3 joint replacement / arthroplasty are counted above but kept OUT of
# the net (too noisy / wrong tier).
SURGICAL_NET_EXCLUDE = {"%joint replacement%", "%arthroplasty%"}
SURGICAL_PATTERNS = [
    p
    for _, pats in SURGICAL_TERMS
    for p in (pats if isinstance(pats, list) else [pats])
    if p not in SURGICAL_NET_EXCLUDE
]

surgical_ids = rsql.text_net_patient_ids(
    session, SOURCE_CONFIG, "surgical_history",
    TEXT_COLUMNS["surgical_history"], SURGICAL_PATTERNS,
)

In [ ]:
surgical_term_counts_df = rsql.term_counts(
    session, SOURCE_CONFIG, "surgical_history",
    TEXT_COLUMNS["surgical_history"], SURGICAL_TERMS, label_width=48,
)
print(f"\nCombined unique (net cell): {len(surgical_ids):,}")
surgical_term_counts_df

### 2.7 Medical History - Value / Source-Category

Tier 1/2 from the ATTR detection framework, plus the heme gate. Phrase-based CTS, bounded ATTR/TTR, no bare `%NEUROPATH%` / `%CARDIOMYOPATH%`.

In [ ]:
MEDICAL_TERMS = [

    # ================================================================
    # ORTHOPEDIC
    # ================================================================

    # Tier 1 - bilateral carpal tunnel syndrome (sig 1)
    ("[ORTHO T1] bilateral carpal tunnel", "%bilateral carpal tunnel%"),
    ("[ORTHO T1] bilateral CTS", "%bilateral CTS%"),
    ("[ORTHO T1] carpal tunnel bilateral", "%carpal tunnel bilateral%"),
    ("[ORTHO T1] CTS bilateral", "%CTS bilateral%"),

    # Tier 1 - spontaneous distal biceps tendon rupture / "Popeye sign" (sig 2)
    ("[ORTHO T1] biceps", "%biceps%"),
    ("[ORTHO T1] popeye", "%popeye%"),

    # Tier 1/2 wide net - any carpal tunnel mention
    ("[ORTHO T1-2] carpal tunnel (space)", "%carpal tunnel%"),
    ("[ORTHO T1-2] carpal_tunnel", "%carpal_tunnel%"),
    ("[ORTHO T1-2] carpal-tunnel", "%carpal-tunnel%"),
    ("[ORTHO T1-2] carpaltunnel", "%carpaltunnel%"),
    ("[ORTHO T1-2] CTS (bounded)", ["% CTS %", "CTS%", "%CTS", "% CTS%", "%CTS %"]),

    # Tier 2 - trigger finger / stenosing tenosynovitis (sig 6)
    ("[ORTHO T2] trigger finger", "%trigger finger%"),
    ("[ORTHO T2] trigger_finger", "%trigger_finger%"),
    ("[ORTHO T2] tenosynov", "%tenosynov%"),

    # Tier 2 - lumbar spinal stenosis (sig 7)
    ("[ORTHO T2] spinal stenosis", "%spinal stenosis%"),
    ("[ORTHO T2] lumbar stenosis", "%lumbar stenosis%"),


    # ================================================================
    # CARDIOLOGY
    # ================================================================

    # Tier 1 - explicit suspicion of infiltrative cardiomyopathy / cardiac
    # amyloid (sig 5)
    ("[CARDIO T1] infiltrative", "%infiltrative%"),
    ("[CARDIO T1] cardiac amyloid", "%cardiac amyloid%"),
    ("[CARDIO T1] amyloid cardiomyopathy", "%amyloid cardiomyopathy%"),

    # Tier 1 - unexplained LV wall thickness / LVH (sig 4)
    ("[CARDIO T1] LVH", "%LVH%"),
    ("[CARDIO T1] left ventricular hypertrophy", "%left ventricular hypertrophy%"),
    ("[CARDIO T1] unexplained LVH", "%unexplained LVH%"),

    # Tier 2 - restrictive / HFpEF / diastolic heart failure family (sig 9, 12)
    ("[CARDIO T2] restrictive cardiomyopathy", "%restrictive cardiomyopathy%"),
    ("[CARDIO T2] HFpEF", "%hfpef%"),
    ("[CARDIO T2] HF preserved", "%heart failure with preserved%"),
    ("[CARDIO T2] diastolic heart failure", "%diastolic heart failure%"),
    ("[CARDIO T2] diastolic dysfunction", "%diastolic dysfunction%"),

    # Tier 2 - low-flow, low-gradient aortic stenosis (sig 12)
    ("[CARDIO T2] aortic stenosis", "%aortic stenosis%"),

    # Tier 2 - atrial fibrillation / flutter (sig 17)
    ("[CARDIO T2] atrial fibrillation", "%atrial fibrillation%"),
    ("[CARDIO T2] atrial fib", "%atrial fib%"),
    ("[CARDIO T2] atrial flutter", "%atrial flutter%"),
    ("[CARDIO T2] AFIB (bounded)", ["% AFIB %", "AFIB%", "%AFIB", "% AFIB%", "%AFIB %"]),

    # Tier 2 - permanent pacemaker or ICD implanted before diagnosis (sig 16)
    ("[CARDIO T2] pacemaker", "%pacemaker%"),

    # Tier 2 - conduction disease: AV block, bundle branch block, sinus node
    ("[CARDIO T2] AV block", "%AV block%"),
    ("[CARDIO T2] atrioventricular block", "%atrioventricular block%"),
    ("[CARDIO T2] bundle branch block", "%bundle branch block%"),
    ("[CARDIO T2] heart block", "%heart block%"),


    # ================================================================
    # NEUROLOGY
    # ================================================================

    # Tier 2 - polyneuropathy / axonal / small fiber neuropathy (sig 1, 2, 8)
    # (no bare "%neuropath%" - too broad/noisy on its own)
    ("[NEURO T2] polyneuropath", "%polyneuropath%"),
    ("[NEURO T2] axonal neuropath", "%axonal neuropath%"),
    ("[NEURO T2] small fiber", "%small fiber%"),
    ("[NEURO T2] small-fiber", "%small-fiber%"),
    ("[NEURO T2] small-fibre", "%small-fibre%"),

    # Tier 1 - CIDP / amyloid neuropathy / ATTR neuropathy (sig 9, 20)
    ("[NEURO T1] CIDP", "%CIDP%"),
    ("[NEURO T1] amyloid neuropath", "%amyloid neuropath%"),
    ("[NEURO T1] ATTR neuropath", "%ATTR neuropath%"),

    # Tier 2 - orthostatic hypotension / autonomic neuropathy (sig 12)
    ("[NEURO T2] orthostatic", "%orthostatic%"),
    ("[NEURO T2] autonomic neuropath", "%autonomic neuropath%"),

    # Tier 2 - erectile dysfunction. NOTE: this is Tier 3 on the Neurology
    # sheet's own row, but Tier 2 on the Urology sheet (which the source
    # framework says should "own" this signal at the fusion layer to avoid
    # double-scoring). Tagged/counted here at Urology's Tier 2, per that rule.
    ("[NEURO T2] erectile", "%erectile%"),


    # ================================================================
    # GASTROENTEROLOGY - Tier 2
    # ================================================================

    # Tier 2 - gastroparesis / early satiety / delayed gastric emptying (sig 1)
    ("[GI T2] gastroparesis", "%gastroparesis%"),
    ("[GI T2] early satiety", "%early satiety%"),
    ("[GI T2] postprandial", "%postprandial%"),
    ("[GI T2] post-prandial", "%post-prandial%"),
    ("[GI T2] delayed gastric emptying", "%delayed gastric emptying%"),
    ("[GI T2] delayed gastric", "%delayed gastric%"),

    # Tier 2 - alternating constipation and diarrhea (sig 2)
    ("[GI T2] alternating constipation", "%alternating constipation%"),
    ("[GI T2] alternating diarrhea", "%alternating diarrhea%"),
    ("[GI T2] constipation and diarrhea", "%constipation and diarrhea%"),
    ("[GI T2] diarrhea and constipation", "%diarrhea and constipation%"),
    ("[GI T2] alternating bowel", "%alternating bowel%"),

    # Tier 2 - chronic/persistent diarrhea, later becoming continuous (sig 3)
    ("[GI T2] chronic diarrhea", "%chronic diarrhea%"),
    ("[GI T2] persistent diarrhea", "%persistent diarrhea%"),
    ("[GI T2] continuous diarrhea", "%continuous diarrhea%"),


    # ================================================================
    # OPHTHALMOLOGY
    # ================================================================

    # Tier 1 - vitreous amyloid (sig 1)
    ("[EYE T1] vitreous amyloid", "%vitreous amyloid%"),
    ("[EYE T1] amyloid vitreous", "%amyloid vitreous%"),
    ("[EYE T1] vitreous amyloidosis", "%vitreous amyloidosis%"),
    ("[EYE T1] amyloidosis of vitreous", "%amyloidosis of vitreous%"),

    # Tier 2 - secondary open-angle glaucoma (sig 2)
    ("[EYE T2] secondary glaucoma", "%secondary glaucoma%"),


    # ================================================================
    # PATHOLOGY / GENETICS (cross-specialty - see Agent5 Geneticist &
    # Molecular Pathologist; also used by Ortho sig 3, GI sig 6, Nephro sig 3)
    # ================================================================

    # Tier 1 - general amyloid / transthyretin / ATTR mentions
    # the rest of this list and with LAB_TERMS / SURGICAL_TERMS.
    ("[PATH T1] amyloid", "%amyloid%"),
    ("[PATH T1] transthyretin", "%transthyretin%"),
    ("[GENE support] ATTR (bounded)", ["% ATTR %", "ATTR%", "% ATTR%", "%ATTR %", "%ATTR-%"]),
    ("[GENE support] TTR (bounded)", ["% TTR %", "TTR%", "% TTR%", "%TTR %"]),

    # Tier 1 - pathogenic/likely-pathogenic TTR gene variant (sig 1)
    ("[GENE T1] TTR gene", "%TTR gene%"),
    ("[GENE T1] TTR mutation", "%TTR mutation%"),

    # Tier 2 - specific founder mutations by ancestry (sig 2, 3, 4)
    ("[GENE T2] Val30Met", "%Val30Met%"),
    ("[GENE T2] Val50Met", "%Val50Met%"),
    ("[GENE T2] Val122Ile", "%Val122Ile%"),
    ("[GENE T2] V122I", "%V122I%"),
    ("[GENE T2] Thr60Ala", "%Thr60Ala%"),


    # ================================================================
    # HEMATOLOGY / AL SAFETY GATE (flag only - NOT ATTR-positive evidence)
    # ================================================================

    # Tier 1 (gate) - MGUS / monoclonal / immunofixation
    ("[HEME gate] MGUS", "%MGUS%"),
    ("[HEME gate] monoclonal", "%monoclonal%"),
    ("[HEME gate] immunofixation", "%immunofixation%"),
    ("[HEME gate] Bence Jones", "%Bence Jones%"),
    ("[HEME gate] M-spike", "%M-spike%"),
    ("[HEME gate] M spike", "%M spike%"),
    ("[HEME gate] paraprotein", "%paraprotein%"),
    ("[HEME gate] monoclonal band", "%monoclonal band%"),
    ("[HEME gate] clonal plasma cells", "%clonal plasma cells%"),
    ("[HEME gate] plasma cell dyscrasia", "%plasma cell dyscrasia%"),
]

MEDICAL_PATTERNS = [
    p for _, pats in MEDICAL_TERMS for p in (pats if isinstance(pats, list) else [pats])
]

medical_ids = rsql.text_net_patient_ids(
    session, SOURCE_CONFIG, "medical_history",
    TEXT_COLUMNS["medical_history"], MEDICAL_PATTERNS,
)

In [ ]:
medical_term_counts_df = rsql.term_counts(
    session, SOURCE_CONFIG, "medical_history",
    TEXT_COLUMNS["medical_history"], MEDICAL_TERMS, label_width=44,
)
print(f"\nCombined unique (net cell): {len(medical_ids):,}")
medical_term_counts_df

### 2.8 Family History - Condition / FamilyMember

Tier 1/2 FHx. Bounded ATTR/TTR, no bare `%sudden%` / `%neuropath%`.

`INCLUDE_BROAD_HEART_DISEASE` reproduces the broader/noisier variant from the old notebook - flip it to compare counts.

In [ ]:
INCLUDE_BROAD_HEART_DISEASE = False

FAMILY_TERMS = [

    # ================================================================
    # CARDIOLOGY / SUDDEN DEATH - Tier 1-2
    # ================================================================

    # Tier 1 - family history of sudden cardiac death / unexplained death
    ("[CARDIO T1] sudden death", "%sudden death%"),
    ("[CARDIO T1] sudden cardiac", "%sudden cardiac%"),
    ("[CARDIO T1] sudden_cardiac", "%sudden_cardiac%"),
    ("[CARDIO T1] cardiac death", "%cardiac death%"),
    ("[CARDIO T1] unexplained death", "%unexplained death%"),
    ("[CARDIO T1] unexplained cardiac", "%unexplained cardiac%"),

    # Tier 1 - family history of infiltrative / amyloid / familial cardiomyopathy
    ("[CARDIO T1] infiltrative cardiomyopathy", "%infiltrative cardiomyopathy%"),
    ("[CARDIO T1] amyloid cardiomyopathy", "%amyloid cardiomyopathy%"),
    ("[CARDIO T1] familial cardiomyopathy", "%familial cardiomyopathy%"),

    # Tier 2 - family history of early/unexplained heart failure
    ("[CARDIO T2] unexplained heart failure", "%unexplained heart failure%"),
    ("[CARDIO T2] early heart failure", "%early heart failure%"),
    ("[CARDIO T2] heart failure", "%heart failure%"),
    ("[CARDIO T2] heart_failure", "%heart_failure%"),
    ("[CARDIO T2] HFpEF", "%hfpef%"),
    ("[CARDIO T2] restrictive cardiomyopathy", "%restrictive cardiomyopathy%"),
    ("[CARDIO T2] hypertrophic cardiomyopathy", "%hypertrophic cardiomyopathy%"),


    # ================================================================
    # NEUROLOGY (no bare %neuropath% - too broad/noisy on its own)
    # ================================================================

    ("[NEURO T2] polyneuropath", "%polyneuropath%"),
    ("[NEURO T1] amyloid neuropath", "%amyloid neuropath%"),
    ("[NEURO T1] ATTR neuropath", "%ATTR neuropath%"),
    ("[NEURO T2] small fiber", "%small fiber%"),
    ("[NEURO T2] small-fiber", "%small-fiber%"),
    ("[NEURO T1] CIDP", "%CIDP%"),
    ("[NEURO T1] hereditary neuropathy", "%hereditary neuropathy%"),


    # ================================================================
    # ORTHOPEDIC family history (wide net only - sig 4)
    # ================================================================

    ("[ORTHO T1] bilateral carpal tunnel", "%bilateral carpal tunnel%"),
    ("[ORTHO T1] bilateral CTS", "%bilateral CTS%"),
    ("[ORTHO T1-2] carpal tunnel", "%carpal tunnel%"),
    ("[ORTHO T1-2] carpal-tunnel", "%carpal-tunnel%"),
    ("[ORTHO T1-2] carpaltunnel", "%carpaltunnel%"),
    ("[ORTHO T1-2] CTS (bounded)", ["% CTS %", "CTS%", "%CTS", "% CTS%", "%CTS %"]),


    # ================================================================
    # AMYLOID / ATTR / HEREDITARY (cross-specialty, see Agent5 sig 7-8)
    # ================================================================

    ("[PATH T1] amyloid", "%amyloid%"),
    ("[PATH T1] transthyretin", "%transthyretin%"),
    ("[PATH T1] familial amyloid", "%familial amyloid%"),
    ("[PATH T1] hereditary amyloid", "%hereditary amyloid%"),
    ("[GENE T1] TTR gene", "%TTR gene%"),
    ("[GENE T1] TTR mutation", "%TTR mutation%"),
    ("[GENE T2] Val30Met", "%Val30Met%"),
    ("[GENE T2] Val50Met", "%Val50Met%"),
    ("[GENE T2] Val122Ile", "%Val122Ile%"),
    ("[GENE T2] V122I", "%V122I%"),
    ("[GENE T2] Thr60Ala", "%Thr60Ala%"),
    ("[GENE support] ATTR (bounded)", ["% ATTR %", "ATTR%", "% ATTR%", "%ATTR %", "%ATTR-%"]),
    ("[GENE support] TTR (bounded)", ["% TTR %", "TTR%", "% TTR%", "%TTR %"]),
]

if INCLUDE_BROAD_HEART_DISEASE:
    FAMILY_TERMS += [
        ("[CARDIO broad] heart disease", "%heart disease%"),
        ("[CARDIO broad] heart_disease", "%heart_disease%"),
    ]

FAMILY_PATTERNS = [
    p for _, pats in FAMILY_TERMS for p in (pats if isinstance(pats, list) else [pats])
]

family_ids = rsql.text_net_patient_ids(
    session, SOURCE_CONFIG, "family_history",
    TEXT_COLUMNS["family_history"], FAMILY_PATTERNS,
)

In [ ]:
family_term_counts_df = rsql.term_counts(
    session, SOURCE_CONFIG, "family_history",
    TEXT_COLUMNS["family_history"], FAMILY_TERMS, label_width=44,
)
print(f"\nCombined unique (net cell): {len(family_ids):,}")
family_term_counts_df

### 2.9 Clinical Note - Clinical Note Text / NoteType

This is where echo / CMR / EMG narrative and explicit "suspect amyloid" statements live, so it is the highest-value text source for Tier 1 features.

In [ ]:
NOTE_TERMS = [

    # ================================================================
    # GENERAL / CROSS-SPECIALTY - Tier 1 (amyloid biopsy confirmation)
    # ================================================================

    ("[PATH T1] amyloid", "%amyloid%"),
    ("[PATH T1] transthyretin", "%transthyretin%"),
    ("[PATH T1] infiltrative", "%infiltrative%"),
    ("[PATH T1] congo red", "%congo red%"),
    ("[PATH T1] apple-green", "%apple-green%"),
    ("[PATH T1] birefringence", "%birefringence%"),
    ("[GENE support] ATTR (bounded)", ["% ATTR %", "ATTR%", "% ATTR%", "%ATTR %", "%ATTR-%"]),
    ("[GENE support] TTR (bounded)", ["% TTR %", "TTR%", "% TTR%", "%TTR %"]),

    ("[EYE T1] vitreous amyloid", "%vitreous amyloid%"),
    ("[EYE T1] amyloid vitreous", "%amyloid vitreous%"),


    # ================================================================
    # CARDIOLOGY - Tier 1 imaging narrative
    # ================================================================

    # Tier 1 - apical sparing / "cherry on top" strain (sig 1)
    ("[CARDIO T1] apical sparing", "%apical sparing%"),
    ("[CARDIO T1] cherry on top", "%cherry on top%"),
    ("[CARDIO T1] cherry-on-top", "%cherry-on-top%"),
    ("[CARDIO T1] bullseye", "%bullseye%"),

    # Tier 1 - expanded extracellular volume, ECV (sig 2)
    ("[CARDIO T1] extracellular volume", "%extracellular volume%"),
    ("[CARDIO T1] ECV (bounded)", ["% ECV %", "ECV%", "%ECV", "% ECV%", "%ECV %"]),

    # Tier 1 - elevated native T1 mapping (sig 3)
    ("[CARDIO T1] native T1", "%native T1%"),

    # Tier 1 - diffuse LGE (sig 6)
    ("[CARDIO T1] late gadolinium", "%late gadolinium%"),
    ("[CARDIO T1] LGE (bounded)", ["% LGE %", "LGE%", "%LGE", "% LGE%", "%LGE %"]),

    # Tier 1 - difficulty nulling myocardium on inversion recovery MRI (sig 25)
    ("[CARDIO T1] difficulty nulling", "%difficulty nulling%"),

    # Tier 1 - bone scintigraphy / Perugini grade (sig 8)
    ("[CARDIO T1] perugini", "%perugini%"),
    ("[CARDIO T1] pyrophosphate", "%pyrophosphate%"),
    ("[CARDIO T1] PYP scan", "%PYP scan%"),

    # Tier 1 - voltage-mass mismatch (sig 7)
    ("[CARDIO T1] low QRS voltage", "%low QRS voltage%"),
    ("[CARDIO T1] voltage-mass", "%voltage-mass%"),

    # Tier 1 - unexplained LV wall thickness (sig 4)
    ("[CARDIO T1] wall thickness", "%wall thickness%"),
    ("[CARDIO T1] unexplained LVH", "%unexplained LVH%"),
    ("[CARDIO T1] interventricular septal", "%interventricular septal%"),

    # Tier 1 - explicit suspicion of infiltrative cardiomyopathy / cardiac
    # amyloid (sig 5)
    ("[CARDIO T1] cardiac amyloid", "%cardiac amyloid%"),

    # Tier 1 - historical: granular sparkling / speckled myocardium (sig 31)
    ("[CARDIO T1] granular sparkling", "%granular sparkling%"),
    ("[CARDIO T1] speckled myocardium", "%speckled myocardium%"),

    # Tier 2 - reduced global longitudinal strain, GLS (sig 11)
    ("[CARDIO T2] global longitudinal strain", "%global longitudinal strain%"),
    ("[CARDIO T2] reduced GLS", "%reduced GLS%"),

    # Tier 2 - restrictive filling / biatrial enlargement / HFpEF (sig 9, 10, 12)
    ("[CARDIO T2] restrictive filling", "%restrictive filling%"),
    ("[CARDIO T2] biatrial enlargement", "%biatrial enlargement%"),
    ("[CARDIO T2] HFpEF", "%hfpef%"),
    ("[CARDIO T2] heart failure", "%heart failure%"),

    # Tier 2 - low-flow, low-gradient aortic stenosis / TAVR workup (sig 12)
    ("[CARDIO T2] aortic stenosis", "%aortic stenosis%"),
    ("[CARDIO T2] low-flow low-gradient", "%low-flow low-gradient%"),
    ("[CARDIO T2] TAVR", "%TAVR%"),

    # Tier 2 - permanent pacemaker or ICD implanted (sig 16)
    ("[CARDIO T2] pacemaker", "%pacemaker%"),
    ("[CARDIO T2] defibrillator", "%defibrillator%"),
    ("[CARDIO T2] ICD (bounded)", ["% ICD %", "ICD%", "%ICD", "% ICD%", "%ICD %"]),

    # Tier 2 - conduction disease (sig 15)
    ("[CARDIO T2] AV block", "%AV block%"),
    ("[CARDIO T2] atrioventricular block", "%atrioventricular block%"),
    ("[CARDIO T2] bundle branch block", "%bundle branch block%"),
    ("[CARDIO T2] heart block", "%heart block%"),

    # Tier 2 - atrial fibrillation / flutter (sig 17)
    ("[CARDIO T2] atrial fibrillation", "%atrial fibrillation%"),
    # atrial flutter 
    ("[CARDIO T2] atrial flutter", "%atrial flutter%"),


    # ================================================================
    # NEUROLOGY
    # ================================================================

    # Tier 1 - CIDP, especially refractory to IVIG/steroids (sig 9)
    ("[NEURO T1] CIDP", "%CIDP%"),
    ("[NEURO T1] refractory to IVIG", "%refractory to IVIG%"),
    ("[NEURO T1] failed IVIG", "%failed IVIG%"),

    # Tier 1 - neurologist explicitly suspects amyloid neuropathy / ATTR (sig 20)
    ("[NEURO T1] amyloid neuropathy", "%amyloid neuropathy%"),
    ("[NEURO T1] suspect amyloid", "%suspect amyloid%"),
    ("[NEURO T1] rule out ATTR", "%rule out ATTR%"),

    # Tier 2 - polyneuropathy / small fiber neuropathy / IENFD / axonal (sig 1,2,3,8)
    ("[NEURO T2] polyneuropath", "%polyneuropath%"),
    ("[NEURO T2] small fiber neuropathy", "%small fiber neuropathy%"),
    ("[NEURO T2] small-fiber neuropathy", "%small-fiber neuropathy%"),
    ("[NEURO T2] IENFD", "%IENFD%"),
    ("[NEURO T2] intraepidermal nerve", "%intraepidermal nerve%"),
    ("[NEURO T2] axonal neuropathy", "%axonal neuropathy%"),
    ("[NEURO T2] length-dependent", "%length-dependent%"),
    ("[NEURO T2] stocking-glove", "%stocking-glove%"),

    # Tier 2 - orthostatic hypotension confirmed by autonomic testing (sig 12)
    ("[NEURO T2] orthostatic", "%orthostatic%"),

    # Support (cross-signal) - EMG/NCS testing
    ("[NEURO support] nerve conduction", "%nerve conduction%"),   
    ("[NEURO support] EMG (bounded)", ["% EMG %", "EMG%", "%EMG", "% EMG%", "%EMG %"]),


    # ================================================================
    # ORTHOPEDIC
    # ================================================================

    ("[ORTHO T1] bilateral carpal tunnel", "%bilateral carpal tunnel%"),
    ("[ORTHO T1-2] carpal tunnel", "%carpal tunnel%"),
    ("[ORTHO T1] biceps", "%biceps%"),
    ("[ORTHO T1] popeye", "%popeye%"),
    ("[ORTHO T2] trigger finger", "%trigger finger%"),
    ("[ORTHO T2] spinal stenosis", "%spinal stenosis%"),
    ("[ORTHO T2] lumbar stenosis", "%lumbar stenosis%"),


    # ================================================================
    # GASTROENTEROLOGY - Tier 2
    # ================================================================

    ("[GI T2] gastroparesis", "%gastroparesis%"),
    ("[GI T2] early satiety", "%early satiety%"),
    ("[GI T2] postprandial", "%postprandial%"),
    ("[GI T2] delayed gastric", "%delayed gastric%"),
    ("[GI T2] alternating bowel", "%alternating bowel%"),
    ("[GI T2] chronic diarrhea", "%chronic diarrhea%"),
    ("[GI T2] persistent diarrhea", "%persistent diarrhea%"),
    ("[GI T2] continuous diarrhea", "%continuous diarrhea%"),


    # ================================================================
    # NEPHROLOGY - Tier 1/2
    # ================================================================

    ("[NEPHRO T2] proteinuria", "%proteinuria%"),
    ("[NEPHRO T2] albuminuria", "%albuminuria%"),
    ("[NEPHRO T2] declining eGFR", "%declining eGFR%"),
    ("[NEPHRO T2] progressive CKD", "%progressive CKD%"),
    ("[NEPHRO T1] renal biopsy", "%renal biopsy%"),
    ("[NEPHRO T1] renal amyloidosis", "%renal amyloidosis%"),


    # ================================================================
    # UROLOGY - Tier 2
    # ================================================================

    ("[UROL T2] erectile dysfunction", "%erectile dysfunction%"),
    ("[UROL T2] urinary retention", "%urinary retention%"),
    ("[UROL T2] detrusor underactivity", "%detrusor underactivity%"),


    # ================================================================
    # OPHTHALMOLOGY
    # ================================================================

    ("[EYE T1] vitreous amyloidosis", "%vitreous amyloidosis%"),
    ("[EYE T1] vitrectomy", "%vitrectomy%"),
    ("[EYE T2] secondary glaucoma", "%secondary glaucoma%"),
    ("[EYE T2] scalloped pupil", "%scalloped pupil%"),


    # ================================================================
    # GENETICS
    # ================================================================

    ("[GENE T1] TTR gene", "%TTR gene%"),
    ("[GENE T1] TTR mutation", "%TTR mutation%"),
    ("[GENE T1] pathogenic TTR", "%pathogenic TTR%"),
    ("[GENE T2] Val30Met", "%Val30Met%"),
    ("[GENE T2] Val50Met", "%Val50Met%"),
    ("[GENE T2] Val122Ile", "%Val122Ile%"),
    ("[GENE T2] V122I", "%V122I%"),
    ("[GENE T2] Thr60Ala", "%Thr60Ala%"),
    ("[GENE T1] mass spectrometry", "%mass spectrometry%"),


    # ================================================================
    # HEMATOLOGY / AL SAFETY GATE (flag only - NOT ATTR-positive evidence)
    # ================================================================

    ("[HEME gate] MGUS", "%MGUS%"),
    ("[HEME gate] free light chain", "%free light chain%"),
    ("[HEME gate] immunofixation", "%immunofixation%"),
    ("[HEME gate] SPEP", "%SPEP%"),
    ("[HEME gate] UPEP", "%UPEP%"),
    ("[HEME gate] monoclonal protein", "%monoclonal protein%"),
    ("[HEME gate] Bence Jones", "%Bence Jones%"),
    ("[HEME gate] M-spike", "%M-spike%"),
    ("[HEME gate] paraprotein", "%paraprotein%"),
    ("[HEME gate] monoclonal band", "%monoclonal band%"),
    ("[HEME gate] clonal plasma cells", "%clonal plasma cells%"),
    ("[HEME gate] plasma cell dyscrasia", "%plasma cell dyscrasia%"),
]

NOTE_PATTERNS = [
    p for _, pats in NOTE_TERMS for p in (pats if isinstance(pats, list) else [pats])
]

note_ids = rsql.text_net_patient_ids(
    session, SOURCE_CONFIG, "clinical_note",
    TEXT_COLUMNS["clinical_note"], NOTE_PATTERNS,
)

In [ ]:
note_term_counts_df = rsql.term_counts(
    session, SOURCE_CONFIG, "clinical_note",
    TEXT_COLUMNS["clinical_note"], NOTE_TERMS, label_width=44,
)
print(f"\nCombined unique (net cell): {len(note_ids):,}")
note_term_counts_df

### 2.10 Social History - not used in this analysis

Excel marked **no need**. Config stays `enabled=False`. We do not search it or pull it into evidence.

In [ ]:
social_ids = []  # Social History is out of scope 
print("SOCIAL_HISTORY: skipped (not used in this analysis)")

### 2.11 UNION all sources → `ATTR_WIDE_NET_CANDIDATES`

In [ ]:
# SNOMED concept IDs from v2 atoms (best/related only). History rows that have
# a code but no English phrase still enter the wide net.
from pathlib import Path as _Path
_v2_dir = _Path(_mod_dir) / "v2"
if str(_v2_dir) not in sys.path:
    sys.path.insert(0, str(_v2_dir))
import loader as _v2_loader

_ATOM_SNOMED = _v2_loader.load_config().all_active_snomed_codes()
print(f"Active SNOMED concept IDs from atoms: {len(_ATOM_SNOMED)}")

surgical_snomed_ids = rsql.snomed_net_patient_ids(
    session, SOURCE_CONFIG, "surgical_history", _ATOM_SNOMED
)
medical_snomed_ids = rsql.snomed_net_patient_ids(
    session, SOURCE_CONFIG, "medical_history", _ATOM_SNOMED
)
family_snomed_ids = rsql.snomed_net_patient_ids(
    session, SOURCE_CONFIG, "family_history", _ATOM_SNOMED
)

surgical_ids = sorted(set(surgical_ids) | set(surgical_snomed_ids))
medical_ids = sorted(set(medical_ids) | set(medical_snomed_ids))
family_ids = sorted(set(family_ids) | set(family_snomed_ids))
print(
    f"After SNOMED union — surgical {len(surgical_ids):,} | "
    f"medical {len(medical_ids):,} | family {len(family_ids):,}"
)


In [ ]:
net_sources = {
    "CLAIM (ICD/CPT)": claim_ids,
    "LAB": lab_ids,
    "SURGICAL_HISTORY": surgical_ids,
    "MEDICAL_HISTORY": medical_ids,
    "FAMILY_HISTORY": family_ids,
    "CLINICAL_NOTE": note_ids,
}

for label, ids in net_sources.items():
    print(f"{label:22s} {len(ids):>10,}")

all_patient_ids = [pid for ids in net_sources.values() for pid in ids]
all_unique_patient_ids = sorted({p.strip() for p in all_patient_ids if p and p.strip()})

print(f"\nTotal (with duplicates): {len(all_patient_ids):,}")
print(f"Total UNIQUE patients in wide net: {len(all_unique_patient_ids):,}")

In [ ]:
rsql.create_candidates(session, all_unique_patient_ids)
print("Step 2 complete. Next: Step 3 - pull evidence for these patients.")

### Step 3 - Pull evidence for wide-net candidates

**What this is:** for patients in `ATTR_WIDE_NET_CANDIDATES` only, fetch their chart slices and normalize them to the column names the feature engine expects.

**What this is NOT:** combination rules / scoring (that is Step 4).

Normalization applied here:

**One evidence table per source table** — same columns, no reshaping. Only two things change: rows are restricted to Step 2 candidates, and physical names become fixed output names (from `SOURCE_CONFIG`) so Step 4 never sees a client-specific name.

| Output temp table | Mirrors | Output columns |
|-------------------|---------|----------------|
| `ATTR_EVID_PATIENT` | `Census` | `PATIENT_ID`, `SEX`, `DATE_OF_BIRTH`, `AGE_IN_YEARS` (derived), `HOME_CITY`, `HOME_STATE`, `FAMILY_ID` |
| `ATTR_EVID_ENCOUNTER` | `Encounter/Visit` | `VISIT_ID`, `PATIENT_ID`, `ENCOUNTER_DATE` |
| `ATTR_EVID_CLAIM` | `Claim` | `PATIENT_ID`, `VISIT_ID`, `DIAGNOSIS_CODE`, `OTHER_DIAGNOSIS_9`, `OTHER_DIAGNOSIS_10`, `PROCEDURE_CODE`, `PROCEDURE_MODIFIER_1/2/3`, `DIAGNOSIS_TYPE`, `PROVIDER_TYPE`, `SPECIALTY_CODE`, `SPECIALTY_NAME`, `DRG_CODE`, `CLINICAL_NOTES`, `FROM_DATE`, `TO_DATE` |
| `ATTR_EVID_LAB_RESULT` | `Lab` | `LAB_ID`, `VISIT_ID`, `PATIENT_ID`, `LAB_REQUEST_ID`, `LAB_RESULT_ID`, `OBSERVATION_IDENTIFIER`, `OBSERVATION_VALUE`, `RESULT_STATUS`, `LAB_RESULT_NOTE`, `OBSERVATION_DATETIME` |
| `ATTR_EVID_MEDICAL_HISTORY` | `Medical History` | `RECORD_ID`, `VISIT_ID`, `PATIENT_ID`, `SOURCE_CATEGORY`, `VALUE`, `SNOMED`, `SECONDARY_SNOMED`, `EVENT_DATE` |
| `ATTR_EVID_SURGICAL_HISTORY` | `Surgical History` | same shape as medical history |
| `ATTR_EVID_FAMILY_HISTORY` | `Family History` | `RECORD_ID`, `VISIT_ID`, `PATIENT_ID`, `SNOMED`, `CONDITION`, `STATUS`, `FAMILY_MEMBER`, `EVENT_DATE` |
| `ATTR_EVID_CLINICAL_NOTES` | `Clinical Note` | `NOTE_ID`, `VISIT_ID`, `PATIENT_ID`, `NOTE_TYPE`, `NOTE_TEXT`, `NOTE_DATE` |

`Social History` and `Medication` are `enabled: False`, so no evidence table is built for them.

A source table that is disabled or missing still produces its `ATTR_EVID_*` table with the columns above and zero rows, so Step 4 never fails on a missing name.

**Note on claim codes:** the four code columns stay as columns here, exactly as delivered. Step 4 unpivots them into a single `CODE_VALUE` internally, because atom matching is easier against one column than four.

In [ ]:
print("=== Step 3 evidence inventory ===")
evidence_inventory = rsql.build_evidence(session, SOURCE_CONFIG)
print("\nStep 3 complete.")
evidence_inventory

In [ ]:
print("Sample claim rows:")
session.sql(
    """
    SELECT PATIENT_ID, VISIT_ID, DIAGNOSIS_CODE, OTHER_DIAGNOSIS_9, OTHER_DIAGNOSIS_10,
           PROCEDURE_CODE, SPECIALTY_NAME, FROM_DATE, TO_DATE
    FROM ATTR_EVID_CLAIM
    WHERE FROM_DATE IS NOT NULL
    LIMIT 10
    """
).show()

print("Sample note text (truncated):")
session.sql(
    """
    SELECT PATIENT_ID, VISIT_ID, NOTE_TYPE, LEFT(NOTE_TEXT, 200) AS NOTE_PREVIEW
    FROM ATTR_EVID_CLINICAL_NOTES
    WHERE NULLIF(TRIM(NOTE_TEXT), '') IS NOT NULL
    LIMIT 10
    """
).show()

## Step 4 - Specialty Tier agents

**SQL-first engine (v1)** (`specialty_configs/v1/rddt_specialty_sql.py`): atom / feature / tier / shortlist in Snowflake **TEMPORARY** tables.

It does **not** read `Claim.DiagnosisCode` or `Census` names. Step 3 already renamed those into `ATTR_EVID_*` (`PATIENT_ID`, `CODE_VALUE`, `VALUE`, `NOTE_TEXT`). Clinical ICD/CPT/NLP rules live in `specialty_configs/v1/rddt_specialty_config.py`.

**Shortlist rule:** Tier 1 or Tier 2 in **≥2** of Ortho / Cardio / Neuro.


### 4.0 Load specialty modules + knobs

In [ ]:
from pathlib import Path
import sys

# v1 hardcoded engine: specialty_configs/v1/
HERE = Path.cwd()
CANDIDATE_DIRS = [
    HERE / "specialty_configs" / "v1",
    HERE / "v1",
    Path("/tmp/specialty_configs/v1"),
    HERE / "specialty_configs",
]

engine_dir = next(
    (p for p in CANDIDATE_DIRS if (p / "rddt_specialty_sql.py").exists()), None
)
if engine_dir is None:
    raise FileNotFoundError(
        "rddt_specialty_sql.py not found. Searched:\n  "
        + "\n  ".join(str(p) for p in CANDIDATE_DIRS)
    )
if str(engine_dir) not in sys.path:
    sys.path.insert(0, str(engine_dir))
print("Engine modules loaded from:", engine_dir)

import rddt_specialty_config as cfg
import rddt_specialty_sql as sql_eng
importlib.reload(cfg)
importlib.reload(sql_eng)

SHORTLIST_N = cfg.SHORTLIST_N  # None = keep ALL shortlist passers
INCLUDE_VISIT_NOTES = True     # scan Clinical Note text in the warehouse

print("Loaded SQL-first specialty engine (current, hardcoded)")
print("Shortlist specialties:", cfg.SHORTLIST_SPECIALTIES)
print("Min specialties:", cfg.SHORTLIST_MIN_SPECIALTIES)
print("SHORTLIST_N (None=all):", SHORTLIST_N)
print("INCLUDE_VISIT_NOTES:", INCLUDE_VISIT_NOTES)


### 4.1 Run Step 4 in Snowflake (TEMPORARY tables only)

Creates `ATTR_ATOMS`, `ATTR_SPECIALTY_FEATURES`, `ATTR_SPECIALTY_TIERS`, `ATTR_KNOWN_ATTR`, `ATTR_SHORTLIST_LLM`, `ATTR_ARCH2_SUMMARY`.

In [ ]:
out = sql_eng.run_specialty_step4_sql(
    session,
    shortlist_n=SHORTLIST_N,
    include_visit_notes=INCLUDE_VISIT_NOTES,
    skip_filter=False,
)
out["summary"]

### 4.1b Step 4 v2 - config-driven engine (same result, editable JSON instead of Python)

Reads `specialty_configs/v2/atoms/*.json`, `specialty_configs/v2/buckets/*.json`, `specialty_configs/v2/features/*.json` via `loader.py`, compiles each feature's logic tree to SQL via `sql_generator.py`, and writes to separate `ATTR_V2_*` TEMPORARY tables - does not touch or overwrite anything the cell above produced, so both can be compared side by side.


In [ ]:
from pathlib import Path
import sys

HERE = Path.cwd()
CANDIDATE_DIRS = [
    HERE / "specialty_configs" / "v2",
    HERE / "v2",
    Path("/tmp/specialty_configs/v2"),
    HERE / "specialty_configs",
]

v2_dir = next((p for p in CANDIDATE_DIRS if (p / "sql_generator.py").exists()), None)
if v2_dir is None:
    raise FileNotFoundError("sql_generator.py not found. Searched:\n  " + "\n  ".join(str(p) for p in CANDIDATE_DIRS))
if str(v2_dir) not in sys.path:
    sys.path.insert(0, str(v2_dir))
print("specialty_configs (v2) loaded from:", v2_dir)

import sql_generator as sql_eng_v2
importlib.reload(sql_eng_v2)


In [ ]:
out_v2 = sql_eng_v2.run_specialty_step4_v2(session, include_visit_notes=INCLUDE_VISIT_NOTES)
out_v2


### 4.2 Preview results (small pulls only)

In [ ]:
print("=== Output lists (TEMPORARY) ===")
for t in [
    "ATTR_KNOWN_ATTR",
    "ATTR_SHORTLIST_LLM",
    "ATTR_SPECIALTY_TIERS",
    "ATTR_SPECIALTY_FEATURES",
    "ATTR_ATOMS",
    "ATTR_ARCH2_SUMMARY",
]:
    print(f"  {t}: {rsql.count_rows(session, t):,}")

print("\nSummary:")
display(session.table("ATTR_ARCH2_SUMMARY").to_pandas())

print("\nShortlist sample:")
display(session.table("ATTR_SHORTLIST_LLM").to_pandas().head(20))

print("\nTier sample (passers):")
display(
    session.sql(
        "SELECT * FROM ATTR_SPECIALTY_TIERS "
        "WHERE shortlist_pass ORDER BY n_specialties_t12 DESC LIMIT 20"
    ).to_pandas()
)

In [ ]:
print("Patients with Tier 1 in at least 2 specialties:")
session.sql(
    """
    SELECT COUNT(*) AS n_patients_t1_in_at_least_2
    FROM ATTR_SPECIALTY_TIERS
    WHERE (
      IFF(ortho_tier = 1, 1, 0) +
      IFF(cardio_tier = 1, 1, 0) +
      IFF(neuro_tier = 1, 1, 0)
    ) >= 2
    """
).show()

print("Patients with Tier 1 in all three specialties:")
session.sql(
    """
    SELECT COUNT(*) AS n_patients_t1_all_three
    FROM ATTR_SPECIALTY_TIERS
    WHERE ortho_tier = 1 AND cardio_tier = 1 AND neuro_tier = 1
    """
).show()

print("Tier combination breakdown:")
display(
    session.sql(
        """
        SELECT
          COALESCE(TO_VARCHAR(ortho_tier), 'none')  AS ortho_tier,
          COALESCE(TO_VARCHAR(cardio_tier), 'none') AS cardio_tier,
          COALESCE(TO_VARCHAR(neuro_tier), 'none')  AS neuro_tier,
          COUNT(*) AS unique_patients
        FROM ATTR_SPECIALTY_TIERS
        GROUP BY 1, 2, 3
        ORDER BY unique_patients DESC, ortho_tier, cardio_tier, neuro_tier
        """
    ).to_pandas()
)

## Data fetch - chart pull for one tier combination

Set the combination you want to review, then pull each evidence slice for those patients.

In [ ]:
COMBO_FILTER = "ortho_tier = 2 AND cardio_tier = 2 AND neuro_tier IS NULL"

session.sql(
    f"""
    CREATE OR REPLACE TEMPORARY TABLE ATTR_COMBO_PIDS AS
    SELECT PATIENT_ID
    FROM ATTR_SPECIALTY_TIERS
    WHERE {COMBO_FILTER}
    """
).collect()

print(f"Patients in combo: {rsql.count_rows(session, 'ATTR_COMBO_PIDS'):,}")

In [ ]:
combo_slices = {
    "patient": "ATTR_EVID_PATIENT",
    "encounter": "ATTR_EVID_ENCOUNTER",
    "claim": "ATTR_EVID_CLAIM",
    "codes": "ATTR_EVID_CODES_ARCH2",
    "lab": "ATTR_EVID_LAB_ARCH2",
    "medical_history": "ATTR_EVID_MEDICAL_HISTORY",
    "surgical_history": "ATTR_EVID_SURGICAL_HISTORY",
    "family_history": "ATTR_EVID_FAMILY_HISTORY",
    "clinical_notes": "ATTR_EVID_CLINICAL_NOTES",
    "tiers": "ATTR_SPECIALTY_TIERS",
    "feature_hits": "ATTR_SPECIALTY_FEATURES",
}

combo_data = {}
for label, table in combo_slices.items():
    try:
        combo_data[label] = session.sql(
            f"""
            SELECT t.*
            FROM {table} t
            JOIN ATTR_COMBO_PIDS c
              ON TRIM(TO_VARCHAR(t.PATIENT_ID)) = TRIM(TO_VARCHAR(c.PATIENT_ID))
            """
        ).to_pandas()
        print(f"{label:18s} {len(combo_data[label]):>10,} rows")
    except Exception as exc:
        print(f"{label:18s} skipped ({type(exc).__name__})")

combo_data.get("tiers", pd.DataFrame()).head(20)